# 28 — Information-Theoretic Analysis of FIM Sync

The FIM (strange loop) maximises the system's sensitivity to its own collective state.
In information-theoretic terms, this is related to:
1. **Fisher Information** of the order parameter R — how precisely can the system estimate its own sync state?
2. **Mutual Information** between oscillators — does FIM increase inter-oscillator information?
3. **Effective Φ (Integrated Information)** — does FIM increase the system's irreducibility?

If FIM increases Φ, this directly connects the strange loop to IIT's measure of consciousness.

In [ ]:
import numpy as np
import json
from pathlib import Path
from scipy.stats import entropy as kl_div

import sys
sys.path.insert(0, '/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src')
from scpn_quantum_control.bridge.knm_hamiltonian import build_knm_paper27, OMEGA_N_16

N = 16
K_base = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]

def fim_gradient(phases, i):
    n = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases[i] + np.pi) % (2 * np.pi) - np.pi
    sensitivity = 1.0 / (1.0 - R**2 + 0.01)
    return (1.0 / n) * np.sin(phase_diff) * min(sensitivity, 50.0)

def simulate_trajectory(K_scale, fim_lambda, dt=0.02, T=150.0, noise=0.05, seed=42):
    """Full trajectory: returns theta history (last 25%) for analysis."""
    K = K_base * K_scale
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_steps = int(T / dt)
    
    history = []
    for s in range(n_steps):
        diff = theta[None, :] - theta[:, None]
        coupling = np.sum(K * np.sin(diff), axis=1) / N
        dphi = omega + coupling
        if fim_lambda > 0:
            for i in range(N):
                dphi[i] += fim_lambda * fim_gradient(theta, i)
        theta = (theta + dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)) % (2 * np.pi)
        if s >= n_steps * 3 // 4:
            history.append(theta.copy())
    return np.array(history)  # (T_samples, N)

print('Ready.')

## 1. Fisher Information of Order Parameter

In [ ]:
def fisher_information_R(trajectory):
    """Fisher information of the order parameter R from phase trajectory.
    F_R = 1/Var(R) when R is estimated from samples.
    Higher F_R = more precise self-knowledge of sync state."""
    R_series = np.abs(np.mean(np.exp(1j * trajectory), axis=1))
    var_R = np.var(R_series)
    if var_R < 1e-10:
        return float('inf')  # perfectly determined
    return 1.0 / var_R

print(f'{"K":>6} {"λ":>6} {"<R>":>8} {"Var(R)":>10} {"F_R":>10} {"log F_R":>10}')
for K_scale in [5, 10, 12, 16]:
    for lam in [0, 1, 5]:
        traj = simulate_trajectory(K_scale, lam)
        R_series = np.abs(np.mean(np.exp(1j * traj), axis=1))
        mean_R = np.mean(R_series)
        var_R = np.var(R_series)
        f_R = fisher_information_R(traj)
        log_f = np.log10(f_R) if np.isfinite(f_R) else float('inf')
        print(f'{K_scale:6.0f} {lam:6.1f} {mean_R:8.4f} {var_R:10.6f} {f_R:10.1f} {log_f:10.2f}')

## 2. Pairwise Mutual Information Between Oscillators

In [ ]:
def phase_mutual_information(trajectory, n_bins=18):
    """Mean pairwise MI between oscillator phases.
    MI(θ_i; θ_j) = H(θ_i) + H(θ_j) - H(θ_i, θ_j)
    using discretised phase distributions."""
    n_samples, n_osc = trajectory.shape
    bins = np.linspace(0, 2 * np.pi, n_bins + 1)
    
    # Individual entropies
    H_i = np.zeros(n_osc)
    for i in range(n_osc):
        hist, _ = np.histogram(trajectory[:, i], bins=bins, density=True)
        hist = hist / (hist.sum() + 1e-10)
        H_i[i] = -np.sum(hist[hist > 0] * np.log2(hist[hist > 0]))
    
    # Pairwise MI
    MI_pairs = []
    for i in range(n_osc):
        for j in range(i + 1, n_osc):
            hist_2d, _, _ = np.histogram2d(trajectory[:, i], trajectory[:, j],
                                            bins=[bins, bins], density=True)
            hist_2d = hist_2d / (hist_2d.sum() + 1e-10)
            H_ij = -np.sum(hist_2d[hist_2d > 0] * np.log2(hist_2d[hist_2d > 0]))
            mi = H_i[i] + H_i[j] - H_ij
            MI_pairs.append(max(0, mi))  # clip numerical noise
    
    return np.mean(MI_pairs), np.array(MI_pairs)

print(f'{"K":>6} {"λ":>6} {"<R>":>8} {"<MI>":>8} {"max MI":>8} {"MI/H ratio":>10}')
for K_scale in [5, 10, 12, 16]:
    for lam in [0, 1, 5]:
        traj = simulate_trajectory(K_scale, lam)
        R = np.mean(np.abs(np.mean(np.exp(1j * traj), axis=1)))
        mean_mi, mi_arr = phase_mutual_information(traj)
        max_mi = np.max(mi_arr)
        # Ratio: how much of single-oscillator entropy is shared
        H_mean = np.log2(18)  # max entropy for 18 bins
        ratio = mean_mi / H_mean if H_mean > 0 else 0
        print(f'{K_scale:6.0f} {lam:6.1f} {R:8.4f} {mean_mi:8.4f} {max_mi:8.4f} {ratio:10.4f}')

## 3. Effective Integrated Information (Φ approximation)

True Φ (IIT 3.0) is NP-hard. We use a tractable proxy:
Φ_eff = MI(system) - max_bipartition MI(part_1) + MI(part_2)

This measures how much information is lost by cutting the system in two.

In [ ]:
def effective_phi(trajectory, n_bins=12):
    """Approximate integrated information.
    Uses geometric bipartition (first N/2 vs second N/2) as proxy."""
    n_samples, n_osc = trajectory.shape
    half = n_osc // 2
    bins = np.linspace(0, 2 * np.pi, n_bins + 1)
    
    # Total system MI (using circular mean as collective variable for each half)
    z_A = np.mean(np.exp(1j * trajectory[:, :half]), axis=1)
    z_B = np.mean(np.exp(1j * trajectory[:, half:]), axis=1)
    
    phase_A = np.angle(z_A) % (2 * np.pi)
    phase_B = np.angle(z_B) % (2 * np.pi)
    
    # Joint entropy H(A, B)
    hist_2d, _, _ = np.histogram2d(phase_A, phase_B, bins=[bins, bins], density=True)
    hist_2d = hist_2d / (hist_2d.sum() + 1e-10)
    H_AB = -np.sum(hist_2d[hist_2d > 0] * np.log2(hist_2d[hist_2d > 0]))
    
    hist_A, _ = np.histogram(phase_A, bins=bins, density=True)
    hist_A = hist_A / (hist_A.sum() + 1e-10)
    H_A = -np.sum(hist_A[hist_A > 0] * np.log2(hist_A[hist_A > 0]))
    
    hist_B, _ = np.histogram(phase_B, bins=bins, density=True)
    hist_B = hist_B / (hist_B.sum() + 1e-10)
    H_B = -np.sum(hist_B[hist_B > 0] * np.log2(hist_B[hist_B > 0]))
    
    MI_AB = H_A + H_B - H_AB
    
    # Phi_eff = MI between the two halves (how much they know about each other)
    return max(0, float(MI_AB))

print('Effective Φ (integrated information proxy):')
print(f'{"K":>6} {"λ":>6} {"<R>":>8} {"Φ_eff":>8}')
phi_data = []
for K_scale in [0, 2, 5, 8, 10, 12, 14, 16, 20]:
    for lam in [0, 1, 5]:
        traj = simulate_trajectory(K_scale, lam)
        R = np.mean(np.abs(np.mean(np.exp(1j * traj), axis=1)))
        phi = effective_phi(traj)
        phi_data.append({'K': K_scale, 'lambda': lam, 'R': round(R, 4), 'phi': round(phi, 4)})
        print(f'{K_scale:6.0f} {lam:6.1f} {R:8.4f} {phi:8.4f}')

print('\n=== KEY QUESTION ===')
# Does FIM increase Phi at the same R?
for K in [10, 12]:
    phi_0 = [d['phi'] for d in phi_data if d['K'] == K and d['lambda'] == 0][0]
    phi_5 = [d['phi'] for d in phi_data if d['K'] == K and d['lambda'] == 5][0]
    R_0 = [d['R'] for d in phi_data if d['K'] == K and d['lambda'] == 0][0]
    R_5 = [d['R'] for d in phi_data if d['K'] == K and d['lambda'] == 5][0]
    print(f'K={K}: λ=0 → R={R_0:.3f}, Φ={phi_0:.4f}; λ=5 → R={R_5:.3f}, Φ={phi_5:.4f}')
    if phi_5 > phi_0:
        print(f'  FIM INCREASES Φ by {phi_5 - phi_0:.4f} ({100*(phi_5-phi_0)/(phi_0+1e-10):.1f}%)')
    else:
        print(f'  FIM does NOT increase Φ')

In [ ]:
# Save
output = {
    'experiment': 'Information-theoretic analysis of FIM sync',
    'N': N,
    'phi_data': phi_data,
}
out_path = Path('/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/information_theoretic_2026-03-29.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Results saved to {out_path}')